# 03 · Preprocessing — first batch

This notebook prepares the selected model’s basic inputs. Notebook 04 explores additional features; notebook 05 includes the chosen ones in the model.


## Imports

In [ ]:
import json
import sys
import os
from pathlib import Path
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sys.path.insert(0, str(Path('../src').resolve()))
from property_ml.splitting import assign_splits
from property_ml.features import prepare


## Load batch

In [ ]:
DATA = Path('../data')
GROUP = int(os.environ['DATA_API_GROUP'])
files = sorted(DATA.glob(f'group-{GROUP}-*.json'))
if not files:
    raise FileNotFoundError('Collect a batch in notebook 01 first.')
with files[0].open() as file:
    batch = json.load(file)
df = pd.DataFrame(batch['data'])
print(f"Group {GROUP} | batch {batch['batch_number']} | {len(df):,} rows")


## Check the target and split

In [ ]:
valid = df['price'].notna() & df['price'].gt(0)
data = df.loc[valid].copy()
splits = assign_splits(data)
train = data.loc[splits == 'train'].copy()
validation = data.loc[splits == 'validation'].copy()
test = data.loc[splits == 'test'].copy()
print('Train / validation / test:', len(train), len(validation), len(test))


## Transformations

In [ ]:
numeric = ['bed', 'bath', 'acre_lot', 'house_size']
categorical = ['brokered_by', 'street', 'city', 'state', 'zip_code']
train, validation, test = (prepare(frame) for frame in [train, validation, test])
preprocess = ColumnTransformer([
    ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
    ('categorical', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20, max_categories=100), categorical),
])
X_train = preprocess.fit_transform(train[numeric + categorical])
X_validation = preprocess.transform(validation[numeric + categorical])
X_test = preprocess.transform(test[numeric + categorical])
print('Shapes:', X_train.shape, X_validation.shape, X_test.shape)


## Notes

Fit preprocessing on the training split so validation and test data stay held out.
